# 02c — Strategy C: SAM 2 zero-shot baseline

Run Meta's [Segment Anything 2](https://github.com/facebookresearch/sam2) with automatic mask generation on the held-out synthetic pairs and on real Manet/Van Gogh/Rembrandt paintings. Post-process the instance masks into a binary edge map so it's directly comparable to the other strategies.

Prereqs:
```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install git+https://github.com/facebookresearch/sam2.git
```

Checkpoint download: the `sam2` package fetches the checkpoint automatically on first use, or you can pre-download from https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
RAW = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'data' / 'raw'
SYNTH = REPO_ROOT / 'data_notcommitted' / 'dstroke_synth'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke' / 'strategy_C_sam2'
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## Load SAM 2 and build the automatic mask generator

Tuned for many small stroke-like masks: many points per side, lower IoU threshold, small minimum area.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

MODEL_CFG = 'configs/sam2.1/sam2.1_hiera_s.yaml'
CKPT_PATH = 'checkpoints/sam2.1_hiera_small.pt'   # adjust if elsewhere

sam2 = build_sam2(MODEL_CFG, CKPT_PATH, device=DEVICE)
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=64,            # dense grid → more, smaller masks
    pred_iou_thresh=0.75,
    stability_score_thresh=0.85,
    min_mask_region_area=25,
    box_nms_thresh=0.5,
)
print('SAM 2 loaded')

## Run on a held-out synthetic painting (quick sanity)

In [ ]:
val_paintings = sorted((SYNTH / 'val' / 'paintings').glob('*.png'))
sample_path = val_paintings[0]
img = np.array(Image.open(sample_path).convert('RGB'), dtype=np.float32) / 255.0
gt = np.array(Image.open(SYNTH / 'val' / 'edges' / sample_path.name).convert('L'), dtype=np.float32) / 255.0

edge, masks = d.run_sam2_automask(img, mask_generator)
metrics = d.compute_edge_metrics(edge, gt)
print(f'{len(masks)} masks, metrics:', metrics)

fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(img); ax[0].set_title('input')
ax[1].imshow(edge, cmap='gray_r'); ax[1].set_title(f'SAM 2 edge map ({len(masks)} masks)')
ax[2].imshow(gt, cmap='gray_r'); ax[2].set_title('ground truth')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

## Run on all val pairs + real authenticated paintings (one per artist)

Saves per-image edge-map PNG + masks JSON into `outputs/dstroke/strategy_C_sam2/`.

In [ ]:
def process(path: Path, save_sub: Path):
    img = np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0
    edge, masks = d.run_sam2_automask(img, mask_generator)
    save_sub.mkdir(parents=True, exist_ok=True)
    out_edge = save_sub / f'{path.stem}_edge.png'
    Image.fromarray((edge * 255).astype(np.uint8)).save(out_edge)
    return edge, masks

# All val
val_dir_out = OUT / 'val'
for p in tqdm(val_paintings, desc='val'):
    process(p, val_dir_out)

# One image per artist from authenticated folders
for artist in ('manet', 'van_gogh', 'rembrandt'):
    art_dir = RAW / artist / 'authenticated'
    if not art_dir.exists():
        print(f'[skip] no {art_dir}')
        continue
    candidates = [p for ext in ('*.tif', '*.tiff', '*.png', '*.jpg') for p in art_dir.glob(ext)][:3]
    for p in candidates:
        process(p, OUT / 'real' / artist)
print('done')

In [ ]:
# Visual grid on the real authenticated samples
from itertools import islice

rows = []
for artist in ('manet', 'van_gogh', 'rembrandt'):
    art_dir = RAW / artist / 'authenticated'
    if not art_dir.exists(): continue
    p = next(iter(p for ext in ('*.tif', '*.tiff', '*.png', '*.jpg') for p in art_dir.glob(ext)), None)
    if p is None: continue
    img = np.array(Image.open(p).convert('RGB').resize((256, 256)), dtype=np.float32) / 255.0
    edge_path = OUT / 'real' / artist / f'{p.stem}_edge.png'
    if not edge_path.exists(): continue
    edge = np.array(Image.open(edge_path).convert('L'), dtype=np.float32) / 255.0
    rows.append({'input': img, 'C': edge, '_label': f'{artist}: {p.name}'})

if rows:
    fig = d.build_comparison_grid(rows, col_order=('input', 'C'),
                                   col_titles={'C': 'SAM 2 edges'})
    plt.show()